<a href="https://colab.research.google.com/github/anuvishalp/Python_Projects/blob/main/Exercism-146Practices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## Python exercies

1. Implement a RESTful API for tracking IOUs.

        Four roommates have a habit of borrowing money from each other frequently, and have trouble remembering who owes whom, and how much.

        Your task is to implement a simple RESTful API that receives IOUs as POST requests, and can deliver specified summary information via GET requests.

        API Specification
        User object
        {
          "name": "Adam",
          "owes": {
            "Bob": 12.0,
            "Chuck": 4.0,
            "Dan": 9.5
          },
          "owed_by": {
            "Bob": 6.5,
            "Dan": 2.75
          },
          "balance": "<(total owed by other users) - (total owed to other users)>"
        }

Methods
          Description	HTTP Method	URL	Payload Format	Response w/o Payload	Response w/ Payload
          List of user information	GET	/users	{"users":["Adam","Bob"]}	{"users":<List of all User objects>}	{"users":<List of User objects for <users> (sorted by name)}
          Create user	POST	/add	{"user":<name of new user (unique)>}	N/A	<User object for new user>
          Create IOU	POST	/iou	{"lender":<name of lender>,"borrower":<name of borrower>,"amount":5.25}
          Here’s the complete, clean, production‑ready implementation of the IOU‑tracking REST API you described — fully aligned with the spec, deterministic, and easy to extend.


------------>Full Flask implementation (copy‑paste ready)

🚀 Core Idea
Each user has:
      owes → money they owe others
      owed_by → money others owe them
      balance = sum(owed_by) − sum(owes)
IOUs update both users symmetrically.

🧱 Data Model (in‑memory)
          users = {}
Each user stored as:
          {
              "name": "Adam",
              "owes": {},
              "owed_by": {},
              "balance": 0.0
          }
================================================
🧩 Full Flask Implementation

from flask import Flask, request, jsonify
app = Flask(__name__)
users = {}
def calculate_balance(user):
    return sum(user["owed_by"].values()) - sum(user["owes"].values())

def get_user(name):
    return users.get(name)

def create_user(name):
    user = {
        "name": name,
        "owes": {},
        "owed_by": {},
        "balance": 0.0
    }
    users[name] = user
    return user

def update_iou(lender, borrower, amount):
    lender_user = get_user(lender)
    borrower_user = get_user(borrower)

    # Borrower owes lender
    borrower_user["owes"][lender] = borrower_user["owes"].get(lender, 0) + amount
    lender_user["owed_by"][borrower] = lender_user["owed_by"].get(borrower, 0) + amount

    # Netting out if both owe each other
    if lender in borrower_user["owed_by"]:
        cross = min(borrower_user["owed_by"][lender], borrower_user["owes"][lender])
        borrower_user["owed_by"][lender] -= cross
        borrower_user["owes"][lender] -= cross
        if borrower_user["owed_by"][lender] == 0:
            del borrower_user["owed_by"][lender]
        if borrower_user["owes"][lender] == 0:
            del borrower_user["owes"][lender]

    if borrower in lender_user["owes"]:
        cross = min(lender_user["owes"][borrower], lender_user["owed_by"][borrower])
        lender_user["owed_by"][borrower] -= cross
        lender_user["owes"][borrower] -= cross
        if lender_user["owed_by"][borrower] == 0:
            del lender_user["owed_by"][borrower]
        if lender_user["owes"][borrower] == 0:
            del lender_user["owes"][borrower]

    # Recalculate balances
    lender_user["balance"] = calculate_balance(lender_user)
    borrower_user["balance"] = calculate_balance(borrower_user)

    return [lender_user, borrower_user]


@app.route("/users", methods=["GET"])
def list_users():
    req = request.get_json(silent=True)
    if req and "users" in req:
        selected = sorted([users[name] for name in req["users"]], key=lambda u: u["name"])
        return jsonify({"users": selected})
    return jsonify({"users": sorted(users.values(), key=lambda u: u["name"])})


@app.route("/add", methods=["POST"])
def add_user():
    name = request.json["user"]
    user = create_user(name)
    return jsonify(user)


@app.route("/iou", methods=["POST"])
def create_iou():
    lender = request.json["lender"]
    borrower = request.json["borrower"]
    amount = request.json["amount"]

    updated = update_iou(lender, borrower, amount)
    updated_sorted = sorted(updated, key=lambda u: u["name"])
    return jsonify({"users": updated_sorted})


if __name__ == "__main__":
    app.run(debug=True)
================================================
🚀 API Specification

📌 Behavior Notes

          1. Balance calculation -> Always recomputed after each IOU.
          2. Netting out cross‑debts ->If A owes B and B owes A, the smaller amount cancels.
          3. Sorted output ->All user lists must be sorted alphabetically.

📬 Example Requests

Create users
        POST /add
        {"user": "Adam"}

Create IOU
      POST /iou
      {"lender": "Adam", "borrower": "Bob", "amount": 5.25}

List all users
      GET /users
List specific users
       GET /users
      {"users": ["Bob", "Adam"]}